> Deep Think with Confidence (深度思考，自信表达)

- https://jiaweizzhao.github.io/deepconf/
    - https://jiaweizzhao.github.io/deepconf/static/htmls/code_example.html
    - https://jiaweizzhao.github.io/deepconf/static/pdfs/deepconf_arxiv.pdf
    - https://github.com/vllm-project/vllm/pull/23201/
        - `vllm/v1/engine/logprobs.py`
        - `vllm/v1/engine/output_processor.py`
- Deep Think with Confidence (DeepConf) is a **parallel thinking** method that enhances both LLM reasoning performance and efficiency at test time.
    - confidence signals filter weak reasoning traces (online or offline).
    - weighted voting boosts ensemble accuracy
- 对于 gpt-oss token efficiency
    - 1 - 0.49/3.23 = 84.8%
    - aime 30 道题，99.9% 的准确率?
        - gpt-oss 120b, deepconf@512, tail conf (top 10%),
        - 30 * 4096（sample 512），64 次独立实验，取平均
 
----

- CONFIDENCE AS AN INDICATOR OF REASONING QUALITY
- (incentivize) potential of the base models
    - strong mathematical reasoning
    - long-chain-of-thought performance

### CONFIDENCE AS AN INDICATOR OF REASONING QUALITY

- token entropy
    - Given a language model’s predicted token distribution $P_i$ at position $i$, the token entropy is defined as:
        - $H_i=-\sum_j P_i(j)\log P_i(j)$
- **token confidence**
    - We define token confidence $C_i$ as the negative average log-probability of the **top-k** tokens at position $i$
        - $C_i=-\frac1k\sum_{j=1}^k\log P_i(j)$
    - where k denotes the number of top tokens considered. **High confidence corresponds to peaked distributions and greater model certainty, while low confidence indicates uncertainty in token prediction.**
- **Average Trace Confidence**. Token-level metrics require aggregation to assess entire reasoning traces. Following Kang et al. (2025), we employ average trace confidence (also termed selfcertainty) as a trace-level quality measure:
    - $C_{\text{avg}}=\frac1N\sum_{i=1}^NC_i$
    - Figure 2：取值 14-22 之间

- 关于 $C_i$：瞬时的 $P_i$ 越大，其实 $C_i$ 越低 ??
    - 一个更尖锐的概率分布，在 $C_i$  的计算下，确实会得到一个更高的数值。

In [1]:
import torch

In [8]:
top_ks_1 = torch.tensor([0.6, 0.1, 0.05, 0.02])  # Peaked Distribution, 模型非常确定
top_ks_2 = torch.tensor([0.15, 0.14, 0.13, 0.11]) # Flat Distribution - 模型非常不确定
C_i = lambda xs: - torch.mean(torch.log(xs))
C_i(top_ks_1), C_i(top_ks_2)

(tensor(2.4303), tensor(2.0277))

### DEEP THINK WITH CONFIDENCE

- CONFIDENCE MEASUREMENTS: $C_t$
    - Group Confidence： $C_G=\text{avg}(C_{k-n+1},C_{k-n+2},\cdots,C_k)$（seq sequence tokens 长度是 $n$，group size）
        - Bottom 10% Group Confidence.
        - Lowest Group Confidence.
    - Tail Confidence：$C_{\text{tail}}=\text{avg}(C_{N-K+1},C_{N-K+2},\cdots, C_{N-1},C_N)$, last $K$ tokens (total tokens $N$)
- OFFLINE THINKING WITH CONFIDENCE
    - Majority Voting
        - $V(a)=\sum_{t\in T}I(\text{answer}(t)=a)$，一人一票；
    - Confidence-Weighted Majority Voting
        - $V(a)=\sum_{t=T}C_t\cdot I(\text{answer}(t)=a)$
    - Confidence Filtering
- ONLINE THINKING WITH CONFIDENCE

### experiments

- 有潜力求解这类问题的 base model：strong mathematical reasoning and long-chain-of-thought performance,
    - deepseek 8b
    - qwen3-8b
    - gp-oss-20b